<a href="https://colab.research.google.com/github/eldhosekroy/churn_prediction/blob/main/churn_rp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Candidate Churn Prediction Model
================================
This script builds a machine learning model to predict candidate churn
using candidate profile, call log, and executive profile datasets.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import pickle
import warnings
import os
import time
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import Counter
from nltk.tokenize import sent_tokenize

from sklearn.metrics import roc_curve, RocCurveDisplay
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.utils import class_weight
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier

from dotenv import load_dotenv

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("="*80)
print("CANDIDATE CHURN PREDICTION MODEL")
print("="*80)

# =============================================================================
# 1. DATA LOADING
# =============================================================================
print("\n" + "="*80)
print("STEP 1: DATA LOADING")
print("="*80)

# Load datasets
base_dir = os.path.dirname(os.path.abspath(__file__))

input_dir = os.path.join(base_dir, 'data')
output_dir = os.path.join(base_dir, 'outputs')
os.makedirs(output_dir, exist_ok=True)

base_dir = os.getcwd()

input_dir = base_dir

enrolled_file = os.path.join(input_dir, 'Endrolled & registred.xlsx')
crm_file = os.path.join(input_dir, 'CRM-All contacts.xlsx')
notes_file = os.path.join(input_dir, 'All Notes-Contacts_copy.xlsx')

try:
    enrolled = pd.read_excel(enrolled_file)
    crm = pd.read_excel(crm_file)
    notes = pd.read_excel(notes_file)

    print(" All datasets loaded successfully!")
    print(f"\n Dataset Shapes:")
    print(f"   - Candidate Profile: {enrolled.shape[0]} rows, {enrolled.shape[1]} columns")
    print(f"   - All Notes: {notes.shape[0]} rows, {notes.shape[1]} columns")
    print(f"   - CRM: {crm.shape[0]} rows, {crm.shape[1]} columns")

except FileNotFoundError as e:
    print(f" Error loading datasets: {e}")
    print("Please ensure all excel files are in the same directory as model.py.")
    sys.exit(1)

# Display first few rows
print("\n Candidate Profile Sample:")
print(enrolled.head(3))
print("\n All Notes Sample:")
print(notes.head(3))
print("\n CRM Sample:")
print(crm.head(3))

# =============================================================================
# 2. DATA HANDLING & EXPLORATION
# =============================================================================
print("\n" + "="*80)
print("STEP 2: DATA HANDLING & EXPLORATION")
print("="*80)

print("\n Candidate Profile Info:")
print(enrolled.info())
print("\n All Notes Info:")
print(notes.info())
print("\n CRM Info:")
print(crm.info())

# Check for missing values
print("\n Missing Values in Candidate Profile:")
print(enrolled.isnull().sum())
print("\n Missing Values in Notes:")
print(notes.isnull().sum())

# checking duplicates values for primary column
print("\n Duplicate Values in Candidate Profile:")
print(enrolled['Contact Id'].duplicated().sum())

# =============================================================================
# 3. DATA LOADING
# =============================================================================
print("\n" + "="*80)
print("STEP 3: DATA PREPROCESSING")
print("="*80)

# 3.1 Handle Missing Values in Candidate Profile
print("\n Handling Missing Values...")

# fill null values for 'invoice' column
enrolled['Invoice'] = enrolled['Invoice'].fillna('No')
enrolled['Invoice'] = enrolled['Invoice'].replace('yes', 'Yes')

# fill null values for 'program joined' column
enrolled['Program Joined'] = enrolled['Program Joined'].fillna('Not joined')

# fill null values for 'program location' column (mode)
mode_program_location = enrolled['Program Location'].mode()[0]
enrolled['Program Location'] = enrolled['Program Location'].fillna(mode_program_location)

# fill null values for 'experience' column
enrolled['Experience'] = enrolled['Experience'].fillna(0)

# fill null values for 'course' column
enrolled['Course'] = enrolled['Course'].fillna('Not mentioned')

# fill null values for 'lead generated on' column
# Sort the DataFrame by 'Lead Generated on' in ascending order
enrolled_sorted = enrolled.sort_values(by='Created Time', ascending=True)
# Fill null values in 'Lead Generated on' with the previous valid date
enrolled['Lead Generated on'] = enrolled_sorted['Lead Generated on'].ffill()
# Sort back to original index order if needed (optional, often good practice)
enrolled = enrolled.sort_index()

#fill null values for 'batch assigned to' column
enrolled['Batch Assigned to'] = enrolled['Batch Assigned to'].fillna('Not assigned')

#fill null values for 'source of lead' column
mode_source_of_lead = enrolled['Source of lead'].mode()[0]
enrolled['Source of lead'] = enrolled['Source of lead'].fillna(mode_source_of_lead)

#fill null values for 'tag' column
enrolled['Tag'] = enrolled['Tag'].fillna('Not mentioned')

#fill null values for 'description' column
enrolled['Description'] = enrolled['Description'].fillna('Not mentioned')

#fill null values for 'mode of program joined' column
enrolled['Mode of Program Joined'] = enrolled['Mode of Program Joined'].fillna('Not mentioned')

# Fill null values for 'Note Title' in the notes dataset
notes['Note Title'] = notes['Note Title'].fillna('Not available')

# Fill null values for 'Note Content' in the notes dataset
notes['Note Content'] = notes['Note Content'].fillna('Not available')

# 3.2 standardising gender column
enrolled['Gender'] = enrolled['Gender'].replace('MAle', 'Male')

# Standardize the 'Program Joined' column
def standardize_program_joined_text(program_string):
    if pd.isna(program_string) or program_string == 'Not joined':
        return program_string

    program_string = str(program_string)

    # 1. Standardize month names (full to short form, and ensure proper case)
    month_mapping = {
        'January': 'Jan', 'February': 'Feb', 'March': 'Mar', 'April': 'Apr',
        'May': 'May', 'June': 'Jun', 'July': 'Jul', 'August': 'Aug',
        'September': 'Sep', 'October': 'Oct', 'November': 'Nov', 'December': 'Dec'
    }
    for full, short in month_mapping.items():
        program_string = re.sub(r'\b' + full + r'\b', short, program_string, flags=re.IGNORECASE)

    # Ensure month abbreviations are capitalized (e.g., "jan" -> "Jan")
        program_string = re.sub(r'\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\b', lambda x: x.group(0).capitalize(), program_string, flags=0)

    # 2. Standardize common program keywords/abbreviations (case-insensitive search, consistent replacement)
        program_string = re.sub(r'\bprogramme\b', 'Program', program_string, flags=re.IGNORECASE)
    # Fix for 'Enrolleeee' issue: replace 'enroll' or 'enrollee' with 'Enrollee' (word boundary for accurate replacement)
        program_string = re.sub(r'\b(enroll|enrollee)\b', 'Enrollee', program_string, flags=re.IGNORECASE)

    # Specific course abbreviations - these should be uppercase
        program_string = re.sub(r'\bmern stack\b', 'MERN', program_string, flags=re.IGNORECASE)
        program_string = re.sub(r'\bmern\b', 'MERN', program_string, flags=re.IGNORECASE)
        program_string = re.sub(r'\bdata analytics\b', 'DA', program_string, flags=re.IGNORECASE)
        program_string = re.sub(r'\bdata analytic\b', 'DA', program_string, flags=re.IGNORECASE)
        program_string = re.sub(r'\bdata science\b', 'DS', program_string, flags=re.IGNORECASE)
        program_string = re.sub(r'\bdata sci\b', 'DS', program_string, flags=re.IGNORECASE)
        program_string = re.sub(r'\bfull stack\b', 'FS', program_string, flags=re.IGNORECASE)

    # Standardize 'One Month Program' variations
        program_string = re.sub(r'\b(One Month Programme|One Month Program)\b', 'One Month Program', program_string, flags=re.IGNORECASE)

    # Clean up extra spaces
        program_string = ' '.join(program_string.split())

    # Final capitalization for desired format (e.g., "DA May 2024"), then fix abbreviations
        program_string = program_string.title()
        program_string = program_string.replace('Da', 'DA').replace('Ds', 'DS').replace('Mern', 'MERN').replace('Fs', 'FS')
    # Ensure Enrollee keeps its capitalization
        program_string = program_string.replace('Enrollee', 'Enrollee')
        program_string = program_string.replace('One Month Program', 'One Month Program') # Keep as is after title()

        return program_string

# Apply this initial standardization to the 'Program Joined' column
enrolled['Program Joined'] = enrolled['Program Joined'].apply(standardize_program_joined_text)

# Function to standardize course names in the 'Course' column, focusing on preserving degree names
def standardize_course_name(course_string):
    if pd.isna(course_string) or str(course_string).strip().lower() == 'not mentioned':
        return 'NOT MENTIONED'

    course_string = str(course_string).upper()

    # Remove general noise: extra spaces, some punctuation, but be careful not to remove critical parts
    # Allow A-Z, 0-9, spaces, &, and hyphens. Only allow these characters
    course_string = re.sub(r'[^-A-Z0-9\s&]', '', course_string)
    course_string = ' '.join(course_string.split()) # Normalize spaces

    # Standardize common degree and course abbreviations
    course_string = course_string.replace('B.TECH', 'BTECH')
    course_string = course_string.replace('B.E.', 'BE')
    course_string = course_string.replace('M.TECH', 'MTECH')
    course_string = course_string.replace('BSC CS', 'BSC-CS')
    course_string = course_string.replace('MSC CS', 'MSC-CS')
    #course_string = course_string.replace('COMPUTER APPLICATION', 'CA')
    course_string = course_string.replace('BACHELOR OF COMPUTER APPLICATION - COMPUTER APPLICATION', 'BCA')
    # Add BVOC specific standardization
    course_string = course_string.replace('BVOC DATA ANALYTICS AND MACHINE LEARNING', 'B VOC-IT')
    course_string = course_string.replace('BVOC', 'B VOC') # General BVOC

    course_string = course_string.replace('COMPUTER SCIENCE AND ENGINEERING', 'BTECH')
    #course_string = course_string.replace('COMPUTER SCIENCE', 'BTECH')
    #course_string = course_string.replace('INFORMATION TECHNOLOGY', 'BTECH')
    course_string = course_string.replace('DATA ANALYTICS', 'DA')
    course_string = course_string.replace('DATA SCIENCE', 'DS')
    #course_string = course_string.replace('BIG DATA ANALYTICS', 'BDA')
    course_string = course_string.replace('ENGLISH LITERATURE', 'BA')
    course_string = course_string.replace('PLUS TWO', 'PLUS TWO')
    course_string = course_string.replace('DIPLOMA', 'DIPLOMA')
    course_string = course_string.replace('DIPLOMA-NON IT', 'DIPLOMA')

    # Handle common course name variations before general IT/NON-IT replacement
    course_string = course_string.replace('B TECH', 'BTECH') # For 'B TECH'
    course_string = course_string.replace('B_TECH', 'BTECH') # For 'B_TECH'
    course_string = course_string.replace('B_COM', 'BCOM') # For 'B_COM'
    course_string = course_string.replace('CA CA', 'CA')
    course_string = course_string.replace('CA-CA', 'CA') # For 'CA - CA' and 'CA CA'

    # Specific fixes for previous output issues (e.g., missing hyphens)
    course_string = course_string.replace('BTECHIT', 'BTECH')
    course_string = course_string.replace('MSCIT', 'MSC-IT')
    course_string = course_string.replace('BSCIT', 'BSC-IT')
    course_string = course_string.replace('MTECH-IT', 'MTECH')
    course_string = course_string.replace('MTECHIT', 'MTECH')

    # Address remaining specific MSC-CS-DA variations directly after initial replacements
    course_string = course_string.replace('MSC-CS DA', 'MSC-IT')
    course_string = course_string.replace('MSC-CSDA', 'MSC-IT')

    # These replacements should happen after primary acronym standardization
    # This line might create issues for standalone 'IT' if not careful. For now, keep as is as other specific replacements handle degree-IT combinations.
    course_string = course_string.replace(' IT', '-IT') # Handle BTech IT etc. to BTech-IT
    course_string = course_string.replace('NON-IT', '-NON-IT') # Handle BTech-Non IT etc.

    # Normalize hyphens: replace multiple hyphens with single, remove spaces around hyphens
    course_string = re.sub(r'-+', '-', course_string).strip('-')
    course_string = re.sub(r'\s*-\s*', '-', course_string) # Remove spaces around hyphens
    course_string = ' '.join(course_string.split()) # Final space normalization after hyphen cleanup

    # Special handling for numerical and very short entries
    if course_string == '2':
        return 'PLUS TWO'
    if course_string == 'COMP':
        return 'BCA'
    if course_string == 'POST GRADUATION':
        return 'PG'
    if course_string == 'BSC SCIENCE':
        return 'BSC'
    if course_string == 'GRADUATED':
        return 'GRADUATED'
    if course_string == 'DIPLOMA-IT':
        return 'DIPLOMA-IT'
    if course_string == 'DIPLOMA-NON-IT':
        return 'DIPLOMA'
    if course_string == 'DIPLOMA':
        return 'DIPLOMA'
    if course_string == 'INCOLLEGE' or course_string == 'IN_COLLEGE' or course_string == 'STUDENT': # Standardize IN_COLLEGE and convert 'STUDENT' if found as a course
        return 'UNSPECIFIED' # 'STUDENT' is a role, not a course
    if course_string == 'OTHERS':
        return 'OTHERS'

    # Remove words like 'PROGRAM', 'DEGREE', 'BACHELOR', 'MASTER' if they are standalone or don't contribute to specific degree names
    # This part should be less aggressive
    course_string = re.sub(r'\b(BACHELOR|MASTER|DEGREE|PROGRAMME|PROGRAM)\b', '', course_string).strip()
    course_string = re.sub(r'\b(OF|AND)\b', '', course_string).strip() # Remove common connectors

    course_string = ' '.join(course_string.split()) # Final space normalization

    # Map common variations to desired output, e.g., 'BTECH-IT' or 'BTECH'
    if 'BTECH' in course_string and 'NON' not in course_string and 'IT' not in course_string:
        course_string = 'BTECH'
    if 'BCA' in course_string:
        course_string = 'BCA'
    if 'MCA' in course_string:
        course_string = 'MCA'
    if 'BCOM' in course_string:
        course_string = 'BCOM'
    if 'MCOM' in course_string:
        course_string = 'MCOM'
    if 'BA' in course_string and 'DA' not in course_string and 'MBA' not in course_string:
        course_string = 'BA'
    if 'MA' in course_string:
        course_string = 'MA'
    if 'BSC' in course_string and 'NON' not in course_string and 'IT' not in course_string and 'CS' not in course_string and 'COMPUTER SCIENCE' not in course_string:
        course_string = 'BSC'
    if 'MSC' in course_string and 'NON' not in course_string and 'IT' not in course_string and 'CS' not in course_string and 'COMPUTER SCIENCE' not in course_string and 'BIG' not in course_string:
        course_string = 'MSC'
    if 'CSE' in course_string:
        course_string = 'BTECH'
    if 'CS' in course_string and 'BSC' not in course_string and 'MSC' not in course_string:
        course_string = 'BTECH'
    if 'COMPUTER SCIENCE' in course_string and 'BSC' not in course_string and 'MSC' not in course_string:
        course_string = 'BTECH'
    if 'B VOC' in course_string:
        course_string = 'B VOC'
    if 'DIPLOMA'in course_string and 'IT' not in course_string:
        course_string = 'DIPLOMA'


    # Ensure consistency for entries like 'BTECH-IT', 'BSC-IT', etc.
    if course_string == 'BTECH-IT':
         course_string = 'BTECH'
    elif course_string == 'BTECH-NON-IT':
         course_string = 'BTECH'
    elif course_string == 'BACHELOR OF COMPUTER APPLICATION - COMPUTER APPLICATION':
         course_string = 'BCA'
    elif course_string == 'BSC-IT':
        pass # Keep as is
    elif course_string == 'MSC-IT':
        pass
    elif course_string == 'BSCSCIENCE':
         course_string = 'BSC'
    elif course_string == 'BSC COMPUTER SCIENCE':
         course_string = 'BSC-IT'
    elif course_string == 'MSC COMPUTER SCIENCEDA':
         course_string = 'MSC-IT'
    elif course_string == 'MSC COMPUTER SCIENCE DA':
         course_string = 'MSC-IT'
    elif course_string == 'MSC BIG DA':
         course_string = 'MSC-IT'
    elif course_string == 'MBA':
         course_string = 'MBA'
    elif course_string == 'MSC-NON-IT':
         course_string = 'MSC'
    elif course_string == 'BSC-NON-IT':
         course_string = 'BSC'
    elif course_string == 'DIPLOMA-IT':
        pass
    elif course_string == 'DIPLOMA':
        pass
    elif course_string == 'MSC-CS-DA':
         course_string = 'MSC-CS, DA'

    # Final check for empty strings or remaining generic terms
    if not course_string or course_string == 'NAN' or course_string == 'UNKNOWN' or course_string == 'UNSPECIFIED':
        return 'UNSPECIFIED'

    return course_string

# Apply this initial standardization to the 'Program Joined' column
enrolled['Course'] = enrolled['Course'].apply(standardize_course_name)

# fill null values of 'track interested' column
mode_track_interested = enrolled['Track Interested'].mode()[0]

enrolled.loc[
    enrolled['Track Interested'].isnull() & (enrolled['Program_Name'] != 'NOT JOINED'),
    'Track Interested'
] = enrolled['Program_Name']

# Fill any remaining null values in 'Track Interested' with the calculated mode
enrolled['Track Interested'] = enrolled['Track Interested'].fillna(mode_track_interested)

# Data type correction
float_columns = enrolled.select_dtypes(include=['float64']).columns

for col in float_columns:
    # Fill NaN values with 0 and then convert to integer
    enrolled[col] = enrolled[col].fillna(0).astype(int)

print("Data types after converting float columns to int:")
enrolled.info()

# 3.3 Notes extraction analysis
# Download necessary NLTK data (run once)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

stop_words = set(stopwords.words('english'))
# Remove 'not' from stopwords to retain negation meaning
stop_words.discard('not')

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if pd.isna(text) or text == 'Not available':
        return []
    text = str(text).lower() # Lowercase
    text = re.sub(r'[^a-z\s]', '', text) # Remove punctuation and numbers
    words = text.split() # Tokenization
    words = [word for word in words if word not in stop_words] # Remove stopwords
    words = [lemmatizer.lemmatize(word) for word in words] # Lemmatization
    return words

notes['cleaned_content'] = notes['Note Content'].apply(preprocess_text)

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# frequent word context analsis
def get_sentences(text):
    if pd.isna(text) or text == 'Not available':
        return []

    text_str = str(text)

    # Replace newline characters with a period and space for better sentence tokenization
    # Handle both '\n' (literal) and '\\n' (escaped) for robustness
    processed_text = re.sub(r'\\n+', '. ', text_str) # For escaped newlines
    processed_text = re.sub(r'\n+', '. ', processed_text)  # For actual newline characters
    # Clean up multiple periods that might result from successive newlines
    processed_text = re.sub(r'\s*\.\s*\.\s*', '. ', processed_text)
    # Ensure there's a space after periods for proper tokenization
    processed_text = re.sub(r'\.(?!\s)', '. ', processed_text)

    # Tokenize the processed text into sentences
    raw_sentences = sent_tokenize(processed_text)

    cleaned_sentences = []
    for sentence in raw_sentences:
        # Clean and strip the sentence for heuristic checks
        # Temporarily remove punctuation for a more accurate 'all-caps single word' check
        cleaned_s_for_check = re.sub(r'[^a-zA-Z0-9\s]', '', sentence).strip()

        # Heuristic to filter out potential 'names' or very short non-informative phrases
        # If a sentence becomes just one word after cleaning and is all uppercase (and not just a single letter), it's likely a name.
        words = cleaned_s_for_check.split()
        if len(words) == 1 and words[0].isupper() and len(words[0]) > 1:
            continue # Filter out single, all-caps words that are likely names (e.g., 'SIJINA')

        # If the original sentence (after stripping) is meaningful, add it
        if sentence.strip():
            cleaned_sentences.append(sentence.strip())

    return cleaned_sentences

notes['sentences'] = notes['Note Content'].apply(get_sentences)

def infer_status_and_reason_from_notes(sentences_list):
    full_note_text = ' '.join(sentences_list).lower()

    # Keywords for 'Joined' status - these should take highest precedence
    joined_keywords = [
        'enrolled', 'paid fees', 'paid the fees', 'will join today', 'joined', 'registered', 'starts program',
        'course started', 'confirmed admission', 'done payment', 'admission confirmed',
        'assessment', 'assessment attended', 'joined today', 'start classes', 'class started'
    ]

    # --- Not Joined Reasons Keywords ---
    competitor_names_list = ['luminar', 'avodha', 'smec', 'liuminar', 'techminds', 'soften', 'techolas', 'lumimar', 'other institute', 'lum', 'excelr', 'freshers job', 'xlr']
    completed_phrases_list = ['already done', 'already completed', 'already did', 'aloready completed', 'doing', 'percuing', 'done with internship'] # Typos considered

    # Keywords for "Join Later"
    join_later_keywords = ['join later', 'will join', 'joining next month', 'joining in', 'joining soon']
    month_names = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

    # New explicit phrases for "Details Shared/Collected (Candidate)"
    candidate_details_phrases = [
        'candidate details shared', 'student details shared', 'profile details shared',
        'contact details shared', 'information shared about candidate',
        'collected candidate details', 'collected student details', 'collected profile details',
        'collected contact details', 'collected information about candidate',
        'details shared with candidate', 'details collected from candidate'
    ]

    # Check for 'Joined' status first
    for keyword in joined_keywords:
        if keyword in full_note_text:
            return 'Joined', 'N/A'

    # --- Join Later Status Logic (moved to higher precedence) ---
    if any(keyword in full_note_text for keyword in join_later_keywords):
        return 'Join Later', 'N/A'
    # Check for 'month name later' pattern
    for month in month_names:
        if re.search(r'\b' + month + r'\s+later\b', full_note_text):
            return 'Join Later', 'N/A'

    # --- Not Joined Reasons Logic ---

    # Priority 1: Specific check for 'Joined Competitor' when combined with 'already done' or 'already completed'
    found_competitor_keyword = any(comp_name in full_note_text for comp_name in competitor_names_list)
    found_completed_phrase = any(comp_phrase in full_note_text for comp_phrase in completed_phrases_list)

    if found_competitor_keyword and found_completed_phrase:
        return 'Not Joined', 'Joined Competitor'

    # Priority 2: Other 'Not Joined' reasons
    not_joined_reasons = {
        'Joined Competitor': [
            'lum', 'luminar', 'avodha', 'excelr', 'smec', 'liuminar', 'techminds', 'soften', 'joined competitor', 'other institute', 'lumimar', 'techolas',
            'freshers job'
        ],
        'Already Working/Internship': [
            'already working', 'already job', 'technopark', 'infopark', 'doing job', 'doing internship', 'working experience', 'placed', 'joined work', 'already internship done', 'currently working',
            'working as a', 'current job', 'previous job', 'passed out', 'got job', 'job offer'
        ],
        'Looking for Job/Internship (Specific Type)': [
            'looking for job', 'job only', 'looking for internship', 'internship only', 'looking for stipended internship',
            'looking for stipend', 'looking for stipended', 'free internship', 'only job', 'only internship', 'stipend internship', 'paid internship'
        ],
        'Not Interested': [
            'not interested', 'no interest', 'not joining', 'not join', 'not intrested for internship', 'not wish to join'
        ],
        'Financial Issue': [
            'fees', 'amount issue', 'money issue', 'expensive', 'cost', 'financial issue', 'fee issue', 'no money', 'low salary', 'salary issue'
        ],
        'Unreachable/Not Connected': [
            'not connected', 'unreachable', 'no network', 'switched off', 'rnr', 'wrong number', 'unanswered', 'no response', 'unresponsive', 'call later', 'not answering',
            'not responding', 'dis call', 'not reachable', 'nc', 'busy', 'call not connected', 'disconnected', 'switch off', 'incoming calls', 'incoming not', 'junk',
            'na', 'invalid', 'rejected', 'not respond', 'out of service', 'wrong number', 'incoming', 'not connecting', 'voice mail', 'voice issue', 'bc', 'wrong no',
            'network issue', 'out of network', 'rhr', 'disconnecting', 'blocked', 'r rn', 'rne', 'not attended', 'callbusy', 'not ringing', 'discall', 'nr', 'r n r'
        ],
        'Decision Pending/Discussing': [
            'decision pending', 'decission pending', 'thinking', 'will inform', 'call back', 'discuss with family', 'discuss with parents', 'will confirm', 'pending decision'
        ],
        'Location Issue': [
            'location issue', 'migrate', 'relocate', 'far away', 'different city', 'distance issue', 'shifted location'
        ],
        'Time/Schedule Conflict': [
            'time issue', 'schedule conflict', 'busy', 'clash', 'no time', 'exam time', 'studies'
        ]
    }

    for reason, keywords in not_joined_reasons.items():
        for keyword in keywords:
            if keyword in full_note_text:
                if reason == 'Looking for Job/Internship (Specific Type)' and any(ni_key in full_note_text for ni_key in ['not interested', 'no interest']):
                    return 'Not Joined', 'Looking for Other Opportunity (Not Interested)'
                return 'Not Joined', reason

    # --- Unclear Status Logic ---
    # Handle "Details Shared/Collected (Candidate)" with explicit phrases
    if any(phrase in full_note_text for phrase in candidate_details_phrases):
        return 'Unclear', 'Details Shared/Collected (Candidate)'


    # If neither joined nor a clear not-joined reason, check for general interest/pending
    if 'interested' in full_note_text or 'enquiring' in full_note_text or 'waiting' in full_note_text or 'more details' in full_note_text:
        return 'Unclear', 'Interested/Pending'

    # Default to 'Unclear' if no definitive status or reason found
    return 'Unclear', 'Other/Unspecified'


# Apply the new function to the notes DataFrame
notes[['inferred_status', 'inferred_reason']] = notes['sentences'].apply(lambda x: pd.Series(infer_status_and_reason_from_notes(x)))

# 3.4 Removing irrelevant columns for churn prediction
columns_to_drop = [
    'First Name',
    'Last Name',
    'Email Opt Out',
    'college Name.id',
    'college Name',
    'Unsubscribed Mode',
    'Unsubscribed Time',
    'City',
    'Mailing State',
    'Mailing Zip',
    'Mailing Country',
    'Whatsapp Number',
    'Region'
]

enrolled = enrolled.drop(columns=columns_to_drop)

# 3.5 Merging the datadets
enrolled_with_notes = pd.merge(enrolled, notes, left_on='Contact Id', right_on='Parent ID.id', how='left', suffixes=('_enrolled', '_note'))

# Now apply the inference function using the correct 'sentences' column.
# Handle NaN values explicitly: if 'sentences' entry is NaN (due to left merge), treat it as 'No Notes Provided'.
enrolled_with_notes[['inferred_status_notes', 'inferred_reason_notes']] = \
    enrolled_with_notes['sentences'].apply(
        lambda x: pd.Series(infer_status_and_reason_from_notes(x))
        if isinstance(x, list) else pd.Series(['Unclear', 'No Notes Provided'])
    )


def aggregate_inferred_status_reason(df_merged):
    # Group by 'Contact Id' and aggregate inferred statuses and reasons
    grouped = df_merged.groupby('Contact Id').agg(
        inferred_statuses=pd.NamedAgg(column='inferred_status_notes', aggfunc=lambda x: list(x.dropna().unique())),
        inferred_reasons=pd.NamedAgg(column='inferred_reason_notes', aggfunc=lambda x: list(x.dropna().unique()))
    ).reset_index()

    final_results = []
    for index, row in grouped.iterrows():
        contact_id = row['Contact Id']
        statuses = row['inferred_statuses']
        reasons = row['inferred_reasons']

        final_status = 'Unclear'
        final_reason = 'Other/Unspecified'

        # Define status precedence
        status_priority = {'Not Joined': 3, 'Join Later': 2, 'Joined': 1, 'Unclear': 4}
        # Filter out 'No Notes Provided' from reasons if it's the only one, or if other specific reasons exist
        filtered_reasons = [r for r in reasons if r not in ['No Notes Provided', 'N/A', 'Other/Unspecified', 'Interested/Pending', 'Details Shared/Collected (Candidate)']]

        # Determine final status based on priority
        current_priority = 0
        for s in statuses:
            if status_priority.get(s, 0) > current_priority:
                final_status = s
                current_priority = status_priority[s]

        # Determine final reason based on the final_status
        if final_status == 'Not Joined':
            # Collect all specific 'Not Joined' reasons
            not_joined_specific_reasons = [r for r in filtered_reasons if r in [
                'Joined Competitor', 'Already Working/Internship', 'Looking for Job/Internship (Specific Type)',
                'Not Interested', 'Financial Issue', 'Unreachable/Not Connected',
                'Decision Pending/Discussing', 'Location Issue', 'Time/Schedule Conflict'
            ]]
            if not_joined_specific_reasons:
                final_reason = ', '.join(sorted(list(set(not_joined_specific_reasons))))
            else:
                # If 'Not Joined' status is inferred but no specific reason, check for generic ones.
                if 'Unspecified Not Joined Reason' in reasons:
                    final_reason = 'Unspecified Not Joined Reason'
                elif 'Other/Unspecified' in reasons:
                    final_reason = 'Other/Unspecified'
                elif 'No Notes Provided' in reasons and len(filtered_reasons) == 0:
                    final_reason = 'No Notes Provided'
                else:
                    final_reason = 'Unspecified Not Joined Reason'
        elif final_status == 'Join Later':
            join_later_specific_reasons = [r for r in filtered_reasons if r == 'Unspecified Join Later Reason'] # 'N/A' is the common reason for 'Join Later' so filtered_reasons is important
            if join_later_specific_reasons:
                final_reason = ', '.join(sorted(list(set(join_later_specific_reasons))))
            else:
                final_reason = 'N/A' # Typically no specific reasons, just the status
        elif final_status == 'Joined':
            final_reason = 'N/A' # Typically no specific reasons for joining
        elif final_status == 'Unclear':
            unclear_specific_reasons = [r for r in filtered_reasons if r in ['Interested/Pending', 'Details Shared/Collected (Candidate)']]
            if unclear_specific_reasons:
                final_reason = ', '.join(sorted(list(set(unclear_specific_reasons))))
            elif 'No Notes Provided' in reasons and len(filtered_reasons) == 0:
                final_reason = 'No Notes Provided'
            else:
                final_reason = 'Other/Unspecified'

        final_results.append({
            'Contact Id': contact_id,
            'final_inferred_status': final_status,
            'final_inferred_reason': final_reason
        })

    return pd.DataFrame(final_results)

# Apply the aggregation function
aggregated_inferences = aggregate_inferred_status_reason(enrolled_with_notes)

# Drop existing 'final_inferred_status' and 'final_inferred_reason' columns from 'enrolled'
# and any _x suffixed versions if they exist, to avoid merge conflicts from re-running the cell.
for col_name in ['final_inferred_status', 'final_inferred_reason']:
    if col_name in enrolled.columns:
        enrolled = enrolled.drop(columns=[col_name])
    if f'{col_name}_x' in enrolled.columns: # Also drop _x suffixed versions
        enrolled = enrolled.drop(columns=[f'{col_name}_x'])
    if f'{col_name}_y' in enrolled.columns: # Also drop _y suffixed versions
        enrolled = enrolled.drop(columns=[f'{col_name}_y'])

# Merge these aggregated inferences back into the original 'enrolled' DataFrame
enrolled = pd.merge(enrolled, aggregated_inferences, on='Contact Id', how='left')

# =============================================================================
# 4. DATA LOADING
# =============================================================================
print("\n" + "="*80)
print("STEP 4: DEFINE TARGET CHURN")
print("="*80)
# Define conditions for churn
conditions = [
    (enrolled['Program Joined'] != 'Not joined') & (enrolled['Invoice'] == 'Yes'), # Non-churned (Active with Invoice)
    (enrolled['Program Joined'] == 'Not joined') & (enrolled['Invoice'] == 'Yes'), # Churned with Invoice
    (enrolled['Invoice'] == 'No') # Candidates without an invoice, or where invoice status is not 'Yes'
]

# Define choices for churn (0 for non-churn, 1 for churn, -1 for other cases)
choices = [0, 1, -1]

enrolled['churn'] = np.select(conditions, choices, default=-1) # Use -1 to denote cases not explicitly 0 or 1


enrolled_for_model = enrolled[enrolled['churn'].isin([0, 1])].copy()


# =============================================================================
# 5. FEATURE ENGINEERING
# =============================================================================
print("\n" + "="*80)
print("STEP 5: FEATURE ENGINEERING")
print("="*80)
# creating binary columns for reasons
enrolled_for_model['joined_competitor'] = enrolled_for_model['final_inferred_reason'].apply(lambda x: 1 if 'Joined Competitor' in str(x) else 0)
enrolled_for_model['already_working'] = enrolled_for_model['final_inferred_reason'].apply(lambda x: 1 if 'Already Working/Internship' in str(x) else 0)
enrolled_for_model['looking_for_job_internship'] = enrolled_for_model['final_inferred_reason'].apply(lambda x: 1 if 'Looking for Job/Internship (Specific Type)' in str(x) else 0)
enrolled_for_model['not_interested'] = enrolled_for_model['final_inferred_reason'].apply(lambda x: 1 if 'Not Interested' in str(x) else 0)
enrolled_for_model['financial_issue'] = enrolled_for_model['final_inferred_reason'].apply(lambda x: 1 if 'Financial Issue' in str(x) else 0)
enrolled_for_model['unreachable_not_connected'] = enrolled_for_model['final_inferred_reason'].apply(lambda x: 1 if 'Unreachable/Not Connected' in str(x) else 0)
enrolled_for_model['decision_pending'] = enrolled_for_model['final_inferred_reason'].apply(lambda x: 1 if 'Decision Pending/Discussing' in str(x) else 0)
enrolled_for_model['no_notes_provided'] = enrolled_for_model['final_inferred_reason'].apply(lambda x: 1 if 'No Notes Provided' in str(x) else 0)

# Convert boolean columns to int for consistency with other numerical features
enrolled_for_model['Test'] = enrolled_for_model['Test'].astype(int)
enrolled_for_model['Followup Email'] = enrolled_for_model['Followup Email'].astype(int)
enrolled_for_model['Invoice_binary'] = enrolled_for_model['Invoice'].apply(lambda x: 1 if x == 'Yes' else 0)

print("Derived binary features and converted boolean columns added to `enrolled_for_model`.")

print("\n Selecting Features for Modeling...")

# Define feature categories based on available data
categorical_features_doc = [
    'Source of lead',
    'Course',
    'background',
    'role',
    'final_inferred_status',
    'Track Interested',
    'Mode of Program Joined',
    'Batch Assigned to',
   'Gender'
]

numerical_features_doc = [
    'Experience',
    'Semester',
    'Year of Graduation',
    'Test',                    # Converted to int above
    'Followup Email',          # Converted to int above
    'Invoice_binary',          # Derived from 'Invoice'
    'joined_competitor',       # Derived from 'notes' reasons
    'already_working',         # Derived from 'notes' reasons
    'looking_for_job_internship', # Derived from 'notes' reasons
    'not_interested',          # Derived from 'notes' reasons
    'financial_issue',         # Derived from 'notes' reasons
    'unreachable_not_connected', # Derived from 'notes' reasons
    'decision_pending',        # Derived from 'notes' reasons
    'no_notes_provided'        # Derived from 'notes' reasons
]

# Filter to only existing columns in enrolled_for_model
categorical_features = [col for col in categorical_features_doc if col in enrolled_for_model.columns]
numerical_features = [col for col in numerical_features_doc if col in enrolled_for_model.columns]

print(f"   Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"   Numerical features ({len(numerical_features)}): {numerical_features}")

# Combine all selected features
all_features = numerical_features + categorical_features
X = enrolled_for_model[all_features].copy()
y = enrolled_for_model['churn'].copy()

# Apply one-hot encoding to categorical features
X = pd.get_dummies(X, columns=categorical_features, drop_first=True)

# =============================================================================
# 6. DATA SPLITING
# =============================================================================
print("\n" + "="*80)
print("STEP 6: DATA SPLITING")
print("="*80)
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

# =============================================================================
# 7. DATA SCALING
# =============================================================================
print("\n" + "="*80)
print("STEP 7: DATA SCALING")
print("="*80)
# Initialize StandardScaler
scaler = StandardScaler()

# Scale numerical features in X_train
X_train_scaled = X_train.copy()
X_train_scaled[numerical_features] = scaler.fit_transform(X_train[numerical_features])

# Scale numerical features in X_test using the scaler fitted on X_train
X_test_scaled = X_test.copy()
X_test_scaled[numerical_features] = scaler.transform(X_test[numerical_features])


# =============================================================================
# 8. IMBALANCING ANALYSIS AND BALACING TECHNIQUE
# =============================================================================
print("\n" + "="*80)
print("STEP 8: IMBALANCING ANALYSIS AND BALACING TECHNIQUE")
print("="*80)
print("Distribution of Churn in Training Set (before balancing):")
display(y_train.value_counts(normalize=True))

churn_rate_train = y_train.value_counts(normalize=True).get(1, 0) # Get churn rate for class 1
non_churn_rate_train = y_train.value_counts(normalize=True).get(0, 0) # Get non-churn rate for class 0

if churn_rate_train > 0:
    imbalance_ratio = non_churn_rate_train / churn_rate_train
    print(f"\nImbalance Ratio (Non-Churn:Churn) in Training Set: {imbalance_ratio:.2f}:1")
else:
    print("\nNo churn instances in the training set.")

def resample_train_set(X_train, y_train, method='none', random_state=42):
    X_resampled, y_resampled, sample_weights = X_train, y_train, None

    # Convert method to lowercase for consistent comparison
    method = method.lower()

    if method == 'oversample':
        oversampler = RandomOverSampler(random_state=random_state)
        X_resampled, y_resampled = oversampler.fit_resample(X_train, y_train)
        print(f"[DEBUG] Oversample applied. X_resampled shape: {X_resampled.shape}, y_resampled shape: {y_resampled.shape}")
    elif method == 'undersample':
        undersampler = RandomUnderSampler(random_state=random_state)
        X_resampled, y_resampled = undersampler.fit_resample(X_train, y_train)
        print(f"[DEBUG] Undersample applied. X_resampled shape: {X_resampled.shape}, y_resampled shape: {y_resampled.shape}")
    elif method == 'smote':
        smote = SMOTE(random_state=random_state)
        X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
        print(f"[DEBUG] SMOTE applied. X_resampled shape: {X_resampled.shape}, y_resampled shape: {y_resampled.shape}")
    elif method == 'class_weight':
        # For class_weight, we don't resample X_train, but compute sample weights
        # We still use the original X_train, y_train for the model training step
        # The `compute_class_weight` function takes the original y_train.
        weights = class_weight.compute_class_weight(
            class_weight='balanced',
            classes=np.unique(y_train),
            y=y_train
        )
        class_weights_dict = {cls: w for cls, w in zip(np.unique(y_train), weights)}
        # Create sample_weights for each instance in the training set
        sample_weights = y_train.map(class_weights_dict).values
        X_resampled, y_resampled = X_train, y_train # No actual resampling of data
        print(f"[DEBUG] Class Weight applied. No resampling of data. X_resampled shape: {X_resampled.shape}, y_resampled shape: {y_resampled.shape}")

    return X_resampled, y_resampled, sample_weights

def evaluate_balancing_techniques(X_train_scaled, y_train, techniques, random_state=42):
    results = {}
    for name, method in techniques.items():
        print(f"\nEvaluating: {name}")
        precisions, recalls, f1_scores = [], [], []

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

        for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train)):
            X_fold_train, X_fold_val = X_train_scaled.iloc[train_idx], X_train_scaled.iloc[val_idx]
            y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

            # Apply balancing technique
            X_resampled, y_resampled, sample_weights = resample_train_set(X_fold_train, y_fold_train, method=method, random_state=random_state)

            # Initialize Logistic Regression model
            # For 'class_weight' method, pass class_weight directly to the model if sample_weights is None
            if method == 'class_weight' and sample_weights is None:
                model = LogisticRegression(solver='liblinear', random_state=random_state, class_weight='balanced')
            else:
                model = LogisticRegression(solver='liblinear', random_state=random_state)

            # Train the model
            if sample_weights is not None and method == 'class_weight': # Only pass sample_weights if 'class_weight' method generated them
                # For 'class_weight' method, X_resampled, y_resampled are still original X_train, y_train
                model.fit(X_resampled, y_resampled, sample_weight=sample_weights)
            else:
                model.fit(X_resampled, y_resampled)

            y_pred = model.predict(X_fold_val)

            precisions.append(precision_score(y_fold_val, y_pred, zero_division=0))
            recalls.append(recall_score(y_fold_val, y_pred, zero_division=0))
            f1_scores.append(f1_score(y_fold_val, y_pred, zero_division=0))

        results[name] = {
            'Precision': np.mean(precisions),
            'Recall': np.mean(recalls),
            'F1 Score': np.mean(f1_scores)
        }
    return pd.DataFrame(results).T

techniques = {
    'None': 'none',
    'Class Weight': 'class_weight',
    'Oversample': 'oversample',
    'Undersample': 'undersample',
    'SMOTE': 'smote'
}

evaluation_results = evaluate_balancing_techniques(X_train_scaled, y_train, techniques)
display(evaluation_results)

# Select the best performing balance_method based on F1 Score
best_balance_method = evaluation_results['F1 Score'].idxmax()
print(f"\nBest balancing method based on F1 Score: {best_balance_method}")

X_train_fit, y_train_fit, sample_weights = resample_train_set(X_train_scaled, y_train, method=best_balance_method)

print(f"Shape of X_train_fit after applying '{best_balance_method}': {X_train_fit.shape}")
print(f"Shape of y_train_fit after applying '{best_balance_method}': {y_train_fit.shape}")
print("\nDistribution of Churn in X_train_fit:")
display(y_train_fit.value_counts(normalize=True))

# 'sample_weights' will only be non-None if 'class_weight' was the chosen method
# Otherwise, it will be None, and models will not use it explicitly.
if sample_weights is not None:
    print("Sample weights have been generated for class_weight method.")
else:
    print("No explicit sample weights generated (resampling method was applied directly to data).")

# Assertion to ensure resampling occurred if method is 'oversample', 'undersample', or 'smote'
if best_balance_method.lower() in ['oversample', 'undersample', 'smote']:
    original_minority_count = y_train.value_counts()[1]
    original_majority_count = y_train.value_counts()[0]
    resampled_minority_count = y_train_fit.value_counts()[1]
    resampled_majority_count = y_train_fit.value_counts()[0]

    if best_balance_method.lower() == 'smote':
        # SMOTE aims to balance the classes so minority count should become equal to majority count
        assert resampled_minority_count == original_majority_count, f"SMOTE failed: Minority count mismatch. Expected {original_majority_count}, got {resampled_minority_count}"
        assert resampled_majority_count == original_majority_count, f"SMOTE failed: Majority count mismatch. Expected {original_majority_count}, got {resampled_majority_count}"
        print(f"\n[INFO] SMOTE resampling confirmed: Minority class count ({resampled_minority_count}) now equals majority class count ({original_majority_count}).")
    elif best_balance_method.lower() == 'oversample':
        # RandomOverSampler also balances to majority class count by default
        assert resampled_minority_count == original_majority_count, f"Oversample failed: Minority count mismatch. Expected {original_majority_count}, got {resampled_minority_count}"
        assert resampled_majority_count == original_majority_count, f"Oversample failed: Majority count mismatch. Expected {original_majority_count}, got {resampled_majority_count}"
        print(f"\n[INFO] Oversample resampling confirmed: Minority class count ({resampled_minority_count}) now equals majority class count ({original_majority_count}).")
    elif best_balance_method.lower() == 'undersample':
        # RandomUnderSampler reduces majority to minority class count by default
        assert resampled_majority_count == original_minority_count, f"Undersample failed: Majority count mismatch. Expected {original_minority_count}, got {resampled_majority_count}"
        assert resampled_minority_count == original_minority_count, f"Undersample failed: Minority count mismatch. Expected {original_minority_count}, got {resampled_minority_count}"
        print(f"\n[INFO] Undersample resampling confirmed: Majority class count ({resampled_majority_count}) now equals minority class count ({original_minority_count}).")

# =============================================================================
# 9. MODEL TRAINING AND EVALUATION
# =============================================================================
print("\n" + "="*80)
print("STEP 9: MODEL TRAINING AND EVALUATION")
print("="*80)
# Define the models with a fixed random_state for reproducibility
models = {
    'Logistic Regression': LogisticRegression(random_state=42, solver='liblinear'),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(), # KNN does not have a random_state parameter
    'Naive Bayes': GaussianNB(), # GaussianNB does not have a random_state parameter
    'SVM': SVC(random_state=42, probability=True), # probability=True is needed for ROC-AUC
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss') # Suppress warning for use_label_encoder
}

print("Defined the following models:")
for name, model in models.items():
    print(f"- {name}: {model.__class__.__name__}")


model_performance = {}

print("Starting model training and evaluation...")
for name, model in models.items():
    print(f"\n--- Evaluating {name} ---")
    start_time = time.time()

    # Train the model
    # Use sample_weights if provided by the best balancing method (e.g., class_weight)
    if best_balance_method == 'Class Weight' and sample_weights is not None:
        model.fit(X_train_fit, y_train_fit, sample_weight=sample_weights)
    else:
        model.fit(X_train_fit, y_train_fit)

    training_time = time.time() - start_time

    # Make predictions
    y_pred = model.predict(X_test_scaled)

    # Get probability predictions for ROC-AUC (if model supports it)
    y_proba = None
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    elif hasattr(model, 'decision_function'): # For SVM with decision_function
        y_proba = model.decision_function(X_test_scaled)
        # Normalize to probabilities if decision_function output is not already probability-like
        if y_proba is not None:
            y_proba = 1 / (1 + np.exp(-y_proba)) # Sigmoid function

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan # Handle models without predict_proba

    # Store model and metrics
    model_performance[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'training_time': training_time,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'ROC-AUC': roc_auc
    }

    print(f"{name} trained and predictions made. Metrics calculated.")

print("\nModel training and evaluation complete. Displaying results:")

# Create a DataFrame to display results neatly
performance_df = pd.DataFrame.from_dict(model_performance, orient='index')
performance_df = performance_df[['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC', 'training_time']]
performance_df = performance_df.round(3)


# Define tolerance for F1 Score tie-breaking
f1_tolerance = 0.01  # Models within this F1 Score range are considered tied

# Sort models primarily by F1 Score and secondarily by ROC-AUC
ranked_models = performance_df.sort_values(by=['F1 Score', 'ROC-AUC'], ascending=[False, False])

print("\nModels ranked by F1 Score (descending) and ROC-AUC (descending for tie-breaking):")
display(ranked_models)

# Get the top F1 Score
top_f1_score = ranked_models.iloc[0]['F1 Score']

# Identify models that are within the f1_tolerance of the top F1 Score
potential_best_models = ranked_models[
    (ranked_models['F1 Score'] >= top_f1_score - f1_tolerance)
].sort_values(by='ROC-AUC', ascending=False)

# Select the model with the highest ROC-AUC among the potential best models
best_model_name = potential_best_models.index[0]

# Retrieve the actual best model object
best_model = model_performance[best_model_name]['model']

print(f"\nSelected Best Model: {best_model_name}")
print(f"F1 Score: {performance_df.loc[best_model_name]['F1 Score']:.3f}")
print(f"ROC-AUC: {performance_df.loc[best_model_name]['ROC-AUC']:.3f}")


# Suppress UserWarning about use_label_encoder
warnings.filterwarnings("ignore", message="`use_label_encoder` is deprecated. Please use `enable_categorical` instead.")

# =============================================================================
# 10. CROSS-VALIDATION (Overfitting Check)
# =============================================================================
print("\n" + "="*80)
print("STEP 10: CROSS-VALIDATION (Overfitting Check)")
print("="*80)

print("\n Performing 5-Fold Cross-Validation...")
print("-"*70)
print(f"{'Model':<25} {'Train Mean':<12} {'Train Std':<12} {'Test Mean':<12} {'Test Std':<12} {'Overfit?':<10}")
print("-"*70)

cv_results = []
best_model_cv_overfit_name = None
best_model_cv_overfit = None
best_test_mean_f1 = -1 # Initialize with a low score

# Ensure `best_balance_method` is correctly set from previous steps
# For SMOTE, class_weight should generally be None or 1, not 'balanced'
current_balance_method = best_balance_method.lower()

X_cv_train, y_cv_train = X_train_fit, y_train_fit

for name, model in models.items():
    # Instantiate model to ensure fresh training, adapting to balance method
    model_cv = None
    if name == 'Logistic Regression':
        model_cv = LogisticRegression(
            random_state=42,
            solver='liblinear',
            class_weight='balanced' if current_balance_method == 'class_weight' else None
        )
    elif name == 'Decision Tree':
        model_cv = DecisionTreeClassifier(
            random_state=42,
            class_weight='balanced' if current_balance_method == 'class_weight' else None
        )
    elif name == 'Random Forest':
        model_cv = RandomForestClassifier(
            random_state=42,
            n_estimators=100,
            class_weight='balanced' if current_balance_method == 'class_weight' else None
        )
    elif name == 'Gradient Boosting':
        model_cv = GradientBoostingClassifier(random_state=42)
    elif name == 'AdaBoost':
        model_cv = AdaBoostClassifier(random_state=42)
    elif name == 'K-Nearest Neighbors':
        model_cv = KNeighborsClassifier()
    elif name == 'Naive Bayes':
        model_cv = GaussianNB()
    elif name == 'SVM':
        model_cv = SVC(
            random_state=42,
            probability=True,
            class_weight='balanced' if current_balance_method == 'class_weight' else None
        )
    elif name == 'XGBoost':
        # For SMOTE, scale_pos_weight is usually 1, or can be adjusted based on resampled data ratio if needed.
        # Since X_train_fit is already balanced by SMOTE, pos_weight should effectively be 1 (or derived from y_train_fit).
        # The document snippet implies using `scale_pos_weight=pos_weight if balance_method == 'class_weight' else 1`
        # Given current_balance_method is 'smote', this would default to 1.
        model_cv = XGBClassifier(
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss',
            scale_pos_weight=1 # Assuming SMOTE already handled imbalance
        )
    else:
        model_cv = model # Fallback to original model instance

    if model_cv is None: # Skip if model type not recognized/handled
        continue

    # Cross-validation on training set
    # Use X_cv_train and y_cv_train (which are X_train_fit, y_train_fit) for training CV
    train_cv_scores = cross_val_score(model_cv, X_cv_train, y_cv_train, cv=5, scoring='f1', error_score='raise')

    # Cross-validation on test set (generalization check)
    test_cv_scores = cross_val_score(model_cv, X_test_scaled, y_test, cv=5, scoring='f1', error_score='raise')

    # Calculate overfitting gap
    train_mean = train_cv_scores.mean()
    train_std = train_cv_scores.std()
    test_mean = test_cv_scores.mean()
    test_std = test_cv_scores.std()

    overfit_gap = train_mean - test_mean
    is_overfit = " YES" if overfit_gap > 0.1 else " NO"

    cv_results.append({
        'Model': name,
        'Train_Mean': train_mean,
        'Train_Std': train_std,
        'Test_Mean': test_mean,
        'Test_Std': test_std,
        'Overfit_Gap': overfit_gap,
        'Is_Overfit': is_overfit
    })

    print(f"{name:<25} {train_mean:<12.4f} {train_std:<12.4f} {test_mean:<12.4f} {test_std:<12.4f} {is_overfit:<10}")

    # Track best model based on test_mean F1 for cross-validation
    if test_mean > best_test_mean_f1:
        best_test_mean_f1 = test_mean
        best_model_cv_overfit_name = name
        best_model_cv_overfit = model_cv # Store the instantiated model

print("-"*70)

cv_results_df = pd.DataFrame(cv_results)
print("\n Cross-Validation Summary (Sorted by Test Mean F1 Score):")
display(cv_results_df.sort_values('Test_Mean', ascending=False))

print(f"\nBest Model from Cross-Validation for Overfitting Check: {best_model_cv_overfit_name} (Test Mean F1: {best_test_mean_f1:.4f})")


# Use performance_df instead of results_df from the example
results_df = performance_df.copy().reset_index().rename(columns={'index': 'Model'})

# Select top 3 models for tuning based on F1 score
top_models = results_df.sort_values(by='F1 Score', ascending=False).head(3)['Model'].tolist()
print(f"\n Top 3 models for tuning: {top_models}")

# Get the best model name from evaluation (Logistic Regression based on earlier F1/ROC-AUC tie-breaking)
best_model_name_initial = ranked_models.index[0] # From previous step's ranked_models

# Force preference for probability-calibrated ensemble models for smooth UI gauge
# This logic ensures that if Logistic Regression (or other simpler models) was initially best,
# we still try to tune a more complex, well-performing model if available.
force_ensemble_preference = False # Set to True if you want to force this preference

if force_ensemble_preference and best_model_name_initial in ['Decision Tree', 'K-Nearest Neighbors', 'SVM', 'Naive Bayes']:
    temp_best_model_name = best_model_name_initial
    for m in ['Random Forest', 'Gradient Boosting', 'XGBoost', 'Logistic Regression']:
        if m in results_df['Model'].values:
            if results_df[results_df['Model'] == m]['F1 Score'].values[0] >= results_df[results_df['Model'] == temp_best_model_name]['F1 Score'].values[0] - f1_tolerance:
                best_model_name = m
                break
    if best_model_name == best_model_name_initial: # If no suitable ensemble found, stick to initial
        best_model_name = best_model_name_initial
else:
    best_model_name = best_model_name_initial

# =============================================================================
# 11. HYPERPARAMETER TUNING
# =============================================================================
print("\n" + "="*80)
print("STEP 11: HYPERPARAMETER TUNING")
print("="*80)
print(f"\n Tuning: {best_model_name}")
print(f"  (Trained on balanced dataset: {X_train_fit.shape[0]} samples with {best_balance_method})")

# Define parameter grids for different models
param_grids = {
    'Random Forest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, 15, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'Gradient Boosting': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7],
        'min_samples_split': [2, 5]
    },
    'XGBoost': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7],
        'subsample': [0.8, 1.0]
    },
    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear']
    }
}

if best_model_name in param_grids:
    param_grid = param_grids[best_model_name]

    # Get the base model instance
    base_model = None
    if best_model_name == 'Random Forest':
        base_model = RandomForestClassifier(
            random_state=42,
            class_weight='balanced' if best_balance_method == 'Class Weight' else None
        )
    elif best_model_name == 'Gradient Boosting':
        base_model = GradientBoostingClassifier(random_state=42)
    elif best_model_name == 'XGBoost':
        # For SMOTE, scale_pos_weight should be 1 as the data is already balanced.
        # If 'Class Weight' was chosen, scale_pos_weight would be handled by class_weight parameter if applicable
        base_model = XGBClassifier(
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss',
            scale_pos_weight=1 # As SMOTE was used, classes are already balanced
        )
    elif best_model_name == 'Logistic Regression':
        base_model = LogisticRegression(
            random_state=42,
            max_iter=1000,
            class_weight='balanced' if best_balance_method == 'Class Weight' else None
        )

    if base_model is not None:
        # Grid Search
        print(f"   Searching through {len(param_grid)} parameter combinations...")

        grid_search = GridSearchCV(
            base_model, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=0
        )

        # Suppress warnings during grid search fit
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            grid_search.fit(X_train_fit, y_train_fit)

        print(f"\n    Best Parameters: {grid_search.best_params_}")
        print(f"    Best CV F1 Score: {grid_search.best_score_:.4f}")

        best_tuned_model = grid_search.best_estimator_
    else:
        print(f"   Could not instantiate base model for {best_model_name}. Using default best model.")
        best_tuned_model = models[best_model_name]
else:
    print(f"   Using default {best_model_name} without hyperparameter tuning, as no param_grid defined.")
    best_tuned_model = models[best_model_name]

print(f"\nHyperparameter tuning complete. Best tuned model is: {best_tuned_model}")


print(f"\nEvaluating the best tuned model ({best_tuned_model.__class__.__name__}) on the unseen test set...")

# Make predictions on the test set
y_pred_tuned = best_tuned_model.predict(X_test_scaled)

# Get probability predictions for ROC-AUC (if model supports it)
y_proba_tuned = None
if hasattr(best_tuned_model, 'predict_proba'):
    y_proba_tuned = best_tuned_model.predict_proba(X_test_scaled)[:, 1]
elif hasattr(best_tuned_model, 'decision_function'):
    y_proba_tuned = best_tuned_model.decision_function(X_test_scaled)
    if y_proba_tuned is not None:
        y_proba_tuned = 1 / (1 + np.exp(-y_proba_tuned)) # Sigmoid function for probability-like scores

# Calculate evaluation metrics
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
precision_tuned = precision_score(y_test, y_pred_tuned, zero_division=0)
recall_tuned = recall_score(y_test, y_pred_tuned, zero_division=0)
f1_tuned = f1_score(y_test, y_pred_tuned, zero_division=0)
roc_auc_tuned = roc_auc_score(y_test, y_proba_tuned) if y_proba_tuned is not None else np.nan

print(f"\n--- Performance of Best Tuned Model ({best_tuned_model.__class__.__name__}) ---")
print(f"Accuracy:  {accuracy_tuned:.4f}")
print(f"Precision: {precision_tuned:.4f}")
print(f"Recall:    {recall_tuned:.4f}")
print(f"F1 Score:  {f1_tuned:.4f}")
print(f"ROC-AUC:   {roc_auc_tuned:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned, zero_division=0))

print("\nConfusion Matrix:")
conf_matrix = confusion_matrix(y_test, y_pred_tuned)
display(pd.DataFrame(conf_matrix, index=['Actual 0', 'Actual 1'], columns=['Predicted 0', 'Predicted 1']))

# =============================================================================
# 12. REGULARISATION
# =============================================================================
print("\n" + "="*80)
print("STEP 12: REGULARISATION")
print("="*80)
# Create regularized versions of top models
regularized_models = {
    'RF (Regularized)': RandomForestClassifier(
        random_state=42,
        n_estimators=100,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt',
        class_weight='balanced' if best_balance_method == 'Class Weight' else None
    ),
    'GB (Regularized)': GradientBoostingClassifier(
        random_state=42,
        n_estimators=100,
        max_depth=5,
        learning_rate=0.05,
        min_samples_split=10,
        subsample=0.8
    )
}

if XGBClassifier is not None:
    regularized_models['XGB (Regularized)'] = XGBClassifier(
        random_state=42,
        n_estimators=100,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='logloss'
    )

print("\n Comparison: Regularized vs Original Models")
print("-"*80)
print(f"{'Model':<30} {'Train F1':<12} {'Test F1':<12} {'Gap':<12} {'Status':<15}")
print("-"*80)
print(f" (All trained on balanced dataset: {X_train_fit.shape[0]} samples)")
print("-"*80)

for name, model in regularized_models.items():
    model.fit(X_train_fit, y_train_fit)

    train_pred = model.predict(X_train_fit)
    test_pred = model.predict(X_test_scaled)

    train_f1 = f1_score(y_train_fit, train_pred)
    test_f1 = f1_score(y_test, test_pred)

    gap = train_f1 - test_f1
    status = " Good" if gap < 0.1 else " Overfitting"

    print(f"{'Model':<30} {train_f1:<12.4f} {test_f1:<12.4f} {gap:<12.4f} {status:<15}")

print("-"*80)

# Final regularized model selection logic
deployment_model_map = {
    'Random Forest': 'RF (Regularized)',
    'Gradient Boosting': 'GB (Regularized)',
    'XGBoost': 'XGB (Regularized)'
}

# Use best_tuned_model from the previous tuning step as the basis
# If it's a tree-based model that was regularized, use the regularized version
if best_model_name in deployment_model_map:
    final_model_name = f"{best_model_name} (Regularized)"
    final_model = regularized_models[deployment_model_map[best_model_name]]
    print(f"\n Final Regularized Model Selected for Deployment: {final_model_name}")
    print(f"  Balancing technique: {best_balance_method}")
    print(f"  Training data: {X_train_fit.shape[0]} balanced samples")
else:
    # If the best model was not a tree-based model that was regularized (e.g., Logistic Regression),
    # we use the best_tuned_model directly as our final model.
    final_model = best_tuned_model
    final_model_name = best_model_name if best_model_name else best_tuned_model.__class__.__name__
    print(f"\n Final Model Selected for Deployment: {final_model_name}")
    print(f"  Balancing technique: {best_balance_method}")
    print(f"  Training data: {X_train_fit.shape[0]} balanced samples")

print(f"  Test evaluation: {X_test_scaled.shape[0]} samples")

# Ensure the final model is fitted on the full balanced training data
final_model.fit(X_train_fit, y_train_fit)

final_pred = final_model.predict(X_test_scaled)
final_proba = final_model.predict_proba(X_test_scaled)[:, 1]

print(f"\n   Model: {final_model_name}")
print(f"   Accuracy:  {accuracy_score(y_test, final_pred):.4f}")
print(f"   F1 Score:  {f1_score(y_test, final_pred):.4f}")
print(f"   ROC-AUC:   {roc_auc_score(y_test, final_proba):.4f}")


print("\nConfusion Matrix:")
conf_matrix = confusion_matrix(y_test, y_pred_tuned)


fpr, tpr, thresholds = roc_curve(y_test, y_proba_tuned)


# Define feature_columns for consistency
feature_columns = X_train_fit.columns.tolist()

# Define final_model_name
final_model_name = final_model.__class__.__name__

# Define label_encoders (as it was not explicitly created, assuming None or empty dict)
label_encoders = {}

# Define balance_method from the previously determined best_balance_method
balance_method = best_balance_method

# Get final predictions and probabilities from the final_model
final_pred = final_model.predict(X_test_scaled)
final_proba = final_model.predict_proba(X_test_scaled)[:, 1]

# Ensure performance_df is available (it was displayed but not explicitly assigned to a variable named 'results_df' before use in summary)
results_df = performance_df.copy().reset_index().rename(columns={'index': 'Model'})

# =============================================================================
# 13. FEATURE IMPORTANCE ANALYSIS
# =============================================================================
print("\n" + "="*80)
print("STEP 13: FEATURE IMPORTANCE ANALYSIS")
print("="*80)
# Get feature importance (for tree-based models or models with feature_importances_)
if hasattr(final_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': final_model.feature_importances_
    }).sort_values('Importance', ascending=False)

    print("\n Top 15 Most Important Features:")
    print(feature_importance.head(15).to_string(index=False))

    # Plot feature importance
    plt.figure(figsize=(12, 8))
    top_features = feature_importance.head(15)
    sns.barplot(x='Importance', y='Feature', data=top_features, palette='viridis')
    plt.title('Top 15 Feature Importance - Churn Prediction', fontsize=14)
    plt.xlabel('Importance Score')
    plt.ylabel('Features')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'feature_importance.png'), dpi=100)
    plt.show()
    print(" Saved: feature_importance.png")

elif hasattr(final_model, 'coef_'):
    # For logistic regression (assuming binary classification)
    # The coefficients represent the log-odds change for a one-unit change in the feature.
    # Magnitude (absolute value) indicates importance.

    # Ensure final_model.coef_ is 2D and get the first row for binary classification
    if final_model.coef_.ndim > 1:
        coef = final_model.coef_[0]
    else:
        coef = final_model.coef_

    feature_importance = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': np.abs(coef)
    }).sort_values('Importance', ascending=False)

    print("\n Top 15 Most Important Features (by coefficient magnitude):")
    print(feature_importance.head(15).to_string(index=False))

    # Plot feature importance
    plt.figure(figsize=(12, 8))
    top_features = feature_importance.head(15)
    sns.barplot(x='Importance', y='Feature', data=top_features, palette='viridis')
    plt.title('Top 15 Feature Importance - Churn Prediction', fontsize=14)
    plt.xlabel('Coefficient Magnitude')
    plt.ylabel('Features')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'feature_importance.png'), dpi=100)
    plt.show()
    print(" Saved: feature_importance.png")

else:
    print("Feature importance not available for this model type.")

# =============================================================================
# 14. REASON EXTRACTION
# =============================================================================
print("\n" + "="*80)
print("STEP 14. REASON EXTRACTION")
print("="*80)
print("\nExtracting suggested churn reasons for churned candidates based on call remarks and feedback...")

# Redefine aggregate_text_by_candidate to work with the notes dataset structure
def aggregate_text_by_candidate(notes_df):
    if 'Parent ID.id' not in notes_df.columns or 'Note Content' not in notes_df.columns:
        return pd.DataFrame(columns=['Parent Id.id', 'All_Note_Content'])

    # Group by 'Parent ID.id' and concatenate 'Note Content'
    agg_notes = notes_df.groupby('Parent ID.id')['Note Content'].apply(
        lambda x: ' '.join(x.dropna().astype(str))
    ).reset_index(name='All_Note_Content')

    # Rename 'Parent ID.id' to 'Contact Id' for easier merging with enrolled_for_model
    agg_notes = agg_notes.rename(columns={'Parent ID.id': 'Contact Id'})
    return agg_notes

def normalize_reason_label(text):
    if not text:
        return None
    normalized = text.strip().lower()
    mappings = {
        'Financial issues': ['financial issues', 'financial', 'payment', 'pay', 'fee', 'emi', 'installment', 'finance'],
        'Not Interested': ['no confirmation', 'lack of interest', 'not interested', 'no interest', 'lost interest', 'not keen', 'disinterested', 'no longer interested'],
        'Joined another institution': ['another instituition', 'joined another', 'joined other', 'admission elsewhere', 'admitted', 'migrated to', 'joined institute', 'joined company', 'enrolled elsewhere'],
        'Unreachable/Not Connected': ['communication gaps', 'no response', 'no pickup', 'unreachable', 'voicemail', 'did not pick', 'not reachable', 'no answer', 'call dropped', 'busy', 'no contact', 'not responding'],
        'Other': ['other', 'unknown', 'unclear']
    }

    for label, keywords in mappings.items():
        if any(k in normalized for k in keywords):
            return label

    normalized_single = normalized.replace('\n', ' ').strip()
    if normalized_single in [lbl.lower() for lbl in mappings]:
        return normalized_single.title()
    return 'Other'

def heuristic_recommendation(reason_label):
    mapping = {
        'Financial issues': 'Offer flexible payment plans, scholarships, or budget-friendly EMI options and follow up on affordability concerns.',
        'Not Interested': 'Re-engage with personalized course benefits, clarify learning outcomes, and offer a second consultation call.',
        'Joined another institution': 'Reach out with retention incentives, compare program strengths, and propose a unique value-added offer.',
        'Unreachable/Not Connected': 'Increase outreach frequency, confirm contact details, and assign a dedicated counselor for follow-up.',
        'Already Working/Internship': 'Highlight career advancement opportunities and specialized training. Emphasize how the program complements existing experience.',
        'Looking for Job/Internship': 'Provide insights into job market trends, showcase success stories, and offer career counseling specific to job/internship search.',
        'Decision Pending/Discussing': 'Follow up with clarity on program benefits, address family concerns, and offer a consultation with an expert.',
        'Location Issue': 'Suggest online/remote learning options or alternative program locations if available.',
        'Time/Schedule Conflict': 'Offer flexible scheduling, part-time options, or self-paced learning modules.',
        'Interested/Pending': 'Provide more detailed information, address specific queries, and maintain regular follow-up to convert interest.',
        'Details Shared/Collected': 'Ensure timely processing of candidate details and initiate contact for next steps.',
        'No Notes Provided': 'Investigate the candidate details further and provide a customized recovery plan based on the latest call context.',
        'Other': 'Investigate the candidate details further and provide a customized recovery plan based on the latest call context.'
    }
    return mapping.get(reason_label, mapping['Other'])

# Modified extract_reason_and_recommendation to use final_inferred_reason more granularly
def extract_reason_and_recommendation(final_inferred_reason_val=None, remarks_text=None):
    suggested_reason = 'Other' # Default reason

    if final_inferred_reason_val not in ['N/A', 'Other/Unspecified', 'No Notes Provided', None]:
        inferred_reason_lower = str(final_inferred_reason_val).lower()

        # Prioritize specific inferred reasons for Suggested_Churn_Reason
        if 'unreachable/not connected' in inferred_reason_lower:
            suggested_reason = 'Unreachable/Not Connected'
        elif 'already working/internship' in inferred_reason_lower:
            suggested_reason = 'Already Working/Internship'
        elif 'looking for job/internship' in inferred_reason_lower: # This covers 'Looking for Job/Internship (Specific Type)'
            suggested_reason = 'Looking for Job/Internship'
        elif 'not interested' in inferred_reason_lower:
            suggested_reason = 'Not Interested'
        elif 'financial issue' in inferred_reason_lower:
            suggested_reason = 'Financial issues'
        elif 'joined competitor' in inferred_reason_lower:
            suggested_reason = 'Joined another institution'
        elif 'decision pending/discussing' in inferred_reason_lower:
            suggested_reason = 'Decision Pending/Discussing'
        elif 'location issue' in inferred_reason_lower:
            suggested_reason = 'Location Issue'
        elif 'time/schedule conflict' in inferred_reason_lower:
            suggested_reason = 'Time/Schedule Conflict'
        elif 'interested/pending' in inferred_reason_lower:
            suggested_reason = 'Interested/Pending'
        elif 'details shared/collected' in inferred_reason_lower: # This covers 'Details Shared/Collected (Candidate)'
            suggested_reason = 'Details Shared/Collected'
        elif 'no notes provided' in inferred_reason_lower: # Explicitly handle 'No Notes Provided' from inferred reason
            suggested_reason = 'No Notes Provided'
        else:
            # If final_inferred_reason_val is present but doesn't match above, keep it as Other
            suggested_reason = 'Other'
    elif remarks_text: # Fallback to parsing raw text if no specific final_inferred_reason
        text = str(remarks_text).lower()

        if any(k in text for k in ['pay', 'payment', 'fee', 'installment', 'emi', 'finance', 'financial']):
             suggested_reason = 'Financial issues'
        elif any(k in text for k in ['no confirmation', 'not interested', 'no interest', 'lack of interest', 'lost interest', 'not keen', 'disinterested', 'no longer interested']):
             suggested_reason = 'Not Interested'
        elif any(k in text for k in ['another instituition', 'joined another', 'joined other', 'admission elsewhere', 'admitted', 'migrated to', 'joined institute', 'joined company', 'enrolled elsewhere']):
             suggested_reason = 'Joined another institution'
        elif any(k in text for k in ['no response', 'no pickup', 'unreachable', 'voicemail', 'did not pick', 'not reachable', 'no answer', 'call dropped', 'busy', 'no contact', 'not responding']):
             suggested_reason = 'Unreachable/Not Connected'
        elif any(k in text for k in ['course not suitable', 'course mismatch', 'course not for me', 'content not relevant']):
             suggested_reason = 'Not Interested'
        else:
             suggested_reason = 'Other'
    else:
        suggested_reason = 'No Notes Provided' # If no inferred reason and no remarks text

    # Get the recommendation based on the determined suggested_reason
    recommendation = heuristic_recommendation(suggested_reason)

    return suggested_reason, recommendation


print("\n" + "=" * 80)
print("STEP 15: EXTRACT CHURN REASONS FOR CANDIDATES")
print("=" * 80)

# Use enrolled_for_model as the base dataframe for churned candidates
base_df_for_reasons = enrolled_for_model.copy()

# Aggregate notes content by Contact Id
aggregated_notes_content = aggregate_text_by_candidate(notes)

# Merge base_df_for_reasons with aggregated notes content
df_with_aggregated_notes = base_df_for_reasons.merge(
    aggregated_notes_content, on='Contact Id', how='left'
)

# Fill NaN in 'All_Note_Content' and 'final_inferred_reason' with appropriate defaults
df_with_aggregated_notes['All_Note_Content'] = df_with_aggregated_notes['All_Note_Content'].fillna('')
df_with_aggregated_notes['final_inferred_reason'] = df_with_aggregated_notes['final_inferred_reason'].fillna('No Notes Provided')

# Apply the reason extraction function, prioritizing final_inferred_reason
df_with_aggregated_notes[['Suggested_Churn_Reason', 'Recommended_Action']] = df_with_aggregated_notes.apply(
    lambda row: extract_reason_and_recommendation(
        final_inferred_reason_val=row['final_inferred_reason'],
        remarks_text=row['All_Note_Content']
    ), axis=1, result_type='expand'
)

# Prepare output for churned candidates
churn_reasons_df = df_with_aggregated_notes.loc[
    df_with_aggregated_notes['churn'] == 1,
    ['Contact Id', 'churn', 'Suggested_Churn_Reason', 'Recommended_Action', 'All_Note_Content']
].copy()

os.makedirs(output_dir, exist_ok=True)

if churn_reasons_df.empty:
    print(' No churn candidates found to suggest reasons for.')
else:
    churn_reasons_df.to_csv(os.path.join(output_dir, 'churn_reasons.csv'), index=False)
    print(' Saved churn reasons to: churn_reasons.csv')

print('\n Churn reason analysis complete!')
print(f" Total candidates analyzed: {len(df_with_aggregated_notes)}")
print("\n Sample reasons assigned:")
print(df_with_aggregated_notes[['Contact Id', 'Suggested_Churn_Reason', 'Recommended_Action']].head(10).to_string(index=False))

# Save full dataset with suggested reasons for inspection
df_with_aggregated_notes.to_csv(os.path.join(output_dir, 'candidates_with_suggested_reasons.csv'), index=False)

# =============================================================================
# 16. MODEL SAVING
# =============================================================================
print("\n" + "="*80)
print("STEP 16: MODEL SAVING")
print("="*80)
# Save the final model and associated artifacts
model_data = {
    'model': final_model,
    'model_name': final_model_name,
    'model_display_name': final_model_name, # Can be set differently if needed
    'scaler': scaler,
    'feature_columns': feature_columns,
    'label_encoders': label_encoders,
    'categorical_features': categorical_features,
    'numerical_features': numerical_features,
    'balance_method': balance_method,
    'training_data_shape': X_train_fit.shape,
    'test_data_shape': X_test_scaled.shape,
    'class_distribution_train': y_train_fit.value_counts().to_dict(),
    'class_distribution_test': y_test.value_counts().to_dict()
}

with open(os.path.join(output_dir, 'churn_prediction_model.pkl'), 'wb') as f:
    pickle.dump(model_data, f)

print(" Model saved to: churn_prediction_model.pkl")

# Save feature importance report if it was generated
if 'feature_importance' in locals():
    feature_importance.to_csv(os.path.join(output_dir, 'feature_importance_report.csv'), index=False)
    print(" Feature importance saved to: feature_importance_report.csv")
else:
    print(" Feature importance report not available for the selected final model.")

# Save model evaluation results
# Ensure 'results_df' is the correct DataFrame containing overall model evaluation results
results_df.to_csv(os.path.join(output_dir, 'model_evaluation_results.csv'), index=False)
print(" Model evaluation results saved to: model_evaluation_results.csv")

# =============================================================================
# 17. SUMMARY
# =============================================================================
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

# Assuming `accuracy_tuned`, `f1_tuned`, `roc_auc_tuned` hold the final model's metrics
# from the 'Evaluate Final Model' section.
# If not, they would need to be recalculated using `final_pred` and `final_proba`.

print(f"""
 CHURN PREDICTION MODEL BUILD COMPLETE!

 Data Loading: Loaded 3 datasets
 Data Handling: Processed missing values and date columns
 Data Preprocessing: Normalized numerical features, encoded categorical features
 Feature Engineering: Created 19+ features from call logs and executive data
 Balancing Technique: Selected {balance_method} using validation F1 score
 Model Evaluation: Compared 9 different ML algorithms
 Cross-Validation: Checked for overfitting using 5-fold CV
 Hyperparameter Tuning: Optimized best model using GridSearchCV
 Regularization: Applied regularization techniques to prevent overfitting

 Output Files:
   - churn_prediction_model.pkl (Trained model)
   - feature_importance_report.csv (Feature importance rankings)
   - model_evaluation_results.csv (Model comparison results)
   - confusion_matrix.png (Confusion matrix visualization)
   - roc_curve.png (ROC curve visualization)
   - feature_importance.png (Feature importance chart)
   - churn_distribution.png (Target variable distribution) (Note: This file was not explicitly generated in the provided notebook, but can be added if needed)

 Best Model: {final_model_name}
 Final Performance:
   - Accuracy: {accuracy_tuned * 100:.2f}%
   - F1 Score: {f1_tuned:.4f}
   - ROC-AUC: {roc_auc_tuned:.4f}

""")

print("="*80)

# To make `churn_distribution.png` available for the summary, let's create a quick placeholder or actual plot if feasible.
# For now, I'll add a placeholder if it doesn't exist, or you can add a cell to generate it earlier.
# Example for churn distribution plot (assuming 'churn' column exists in 'enrolled' or 'enrolled_for_model')
if 'enrolled_for_model' in globals():
    plt.figure(figsize=(6, 4))
    sns.countplot(x='churn', data=enrolled_for_model, palette='coolwarm', hue='churn', legend=False)
    plt.title('Distribution of Churn (0=Non-Churn, 1=Churn)')
    plt.xlabel('Churn Status')
    plt.ylabel('Count')
    plt.savefig(os.path.join(output_dir, 'churn_distribution.png'))
    plt.show()
    print(" Saved: churn_distribution.png")
else:
    print("Skipping churn_distribution.png generation as 'enrolled_for_model' is not available.")

